**Question:** Does an LLM-derived skill prior from free-text profiles help the enriched GRU, especially for cold-start players?

**Assumptions:** Exp C3 unchanged except for the text-skill slot; text scores cover ~86% of players.

**Method:** Replace the neutral slot with real text-skill, confidence and has-text columns (Exp C4).

**Result:** C4 reaches Brier 0.1598 vs 0.1587 for C3: a null result, so text is not used in production.

# 11 — Experiment C4: Enriched GRU + real Text-Skill score (cold-start hypothesis)

Follow-up to `10_enriched_gru.ipynb` (Exp C3 — best model so far, test Brier
0.15869, but its `text_p`/`text_o` head slot was left filled with a neutral constant `0.0`
because the Gemini text-scoring pass over `Trenutci` (free-text player self-descriptions) was
still running concurrently). **That scoring pass is now done** (`../data/trenutci_scores.csv`,
671 players scored, columns `IgracId`, `skill_1_10` (1–10), `ntrp_est`, `confidence` (0–1)).

**This notebook asks one question: does a real, LLM-derived skill PRIOR from free text add
anything to the enriched GRU — especially for cold-start players (thin match history, where ELO
and career stats are least informative)?** Everything else is byte-for-byte Exp C3: same raw
data, same duplicate-and-flip rows, same 80/20 time split, same architecture family, same
hyperparameters/seed, same per-step 9-dim sequence (`OppEloBefore`, `WasH2H` included). **Only
the head's `text_p`/`text_o` slot changes** — from a neutral `0.0` constant to 6 real columns:

**Per player (×2) — replaces Exp C3's 2 neutral `text_p`/`text_o` columns with 6:**
- `text_skill_p` / `text_skill_o` — `skill_1_10`, scaled with **TRAIN-only** mean/std computed
  over scored players who appear in the TRAIN split (never touches TEST, same discipline as
  `ELO_MEAN`/`AGE_MEAN`). Missing (no text score) → filled with `0.0` (= the train mean, i.e.
  "neutral", after centering) — same convention as every other head scalar's debut-match fill.
- `text_conf_p` / `text_conf_o` — the LLM's own `confidence` (already bounded `[0.1, 0.85]`,
  reused raw, no scaling needed). Missing → filled with `0.0` (zero reliability — semantically
  correct: there is no text to be confident about). This lets the model down-weight low-confidence
  text scores itself rather than treating every text score as equally reliable.
- `has_text_p` / `has_text_o` — explicit binary indicator (`1.0` if this player has ANY scored
  `Trenutci` entry, else `0.0`). **Required because coverage is only ~86%** (671 of 779 in-scope
  players) — without this flag the model cannot distinguish "genuinely average text-skill" from
  "no text at all", both of which look identical after the neutral fill.

**Not dropped, not recomputed:** rows for unscored players are KEPT (never dropped) — only the
per-player text columns are neutral-filled + flagged. Every other per-step/head feature (the 9-dim
sequence, ELO/CareerWinRate/Forma/MatchCount/Age/H2H head blocks) is identical to Exp C3.

**Calibration:** isotonic regression only, by design (Exp C3 additionally
ran a temperature-scaling bake-off; this notebook does not repeat that comparison).

**Runtime-safety:** the text-join (vectorized `pd.Series.map`, no per-row loops) + ONE full GRU
retrain were validated end-to-end in `scripts/exp_c4_validate.py` (~110s) *before* this notebook
was assembled — confirmed safe well under the 600s budget that a prior experiment blew through.


In [ ]:
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.use_deterministic_algorithms(False)  # GRU has no deterministic CPU kernel path for this op set

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cpu')
print(f'PyTorch {torch.__version__}, device={device}, threads={torch.get_num_threads()}')


## Step 1 — Load raw export + the P2 engineered frame (identical to Exp C/C2/C3)

In [ ]:
raw = pd.read_csv(DATA_DIR / 'matches_export.csv', parse_dates=['Datum'])
raw['Datum'] = pd.to_datetime(raw['Datum']).dt.tz_localize(None)
raw = raw.sort_values(['Datum', 'SusretId']).reset_index(drop=True)

sym = pd.read_parquet(DATA_DIR / 'features.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)

print(f'raw matches: {raw.shape}')
print(f'sym (duplicate-and-flip) frame: {sym.shape}')
print(f"Split date (from P2): {TAGS['split_date']}, train matches: {TAGS['n_matches_train']:,}, "
      f"test matches: {TAGS['n_matches_test']:,}")
sym[['SusretId', 'Datum', 'PlayerId', 'OpponentId', 'Label', 'Perspective', 'Split']].head()


## Step 2 — Long-format player-match log, extended with `OppEloBefore` (identical to Exp C3)

In [ ]:
common_cols = ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'LigaTip', 'LigaKoeficjent', 'Lokacija']

winner_rows = raw[common_cols + [
    'WinnerId', 'LoserId', 'WinnerEloBefore', 'LoserEloBefore',
    'W_Set1', 'W_Set2', 'W_Set3', 'L_Set1', 'L_Set2', 'L_Set3',
]].rename(columns={
    'WinnerId': 'IgracId', 'LoserId': 'OpponentId',
    'WinnerEloBefore': 'MyEloBefore', 'LoserEloBefore': 'OppEloBefore',
    'W_Set1': 'MySet1', 'W_Set2': 'MySet2', 'W_Set3': 'MySet3',
    'L_Set1': 'OppSet1', 'L_Set2': 'OppSet2', 'L_Set3': 'OppSet3',
}).assign(Won=1)

loser_rows = raw[common_cols + [
    'LoserId', 'WinnerId', 'LoserEloBefore', 'WinnerEloBefore',
    'L_Set1', 'L_Set2', 'L_Set3', 'W_Set1', 'W_Set2', 'W_Set3',
]].rename(columns={
    'LoserId': 'IgracId', 'WinnerId': 'OpponentId',
    'LoserEloBefore': 'MyEloBefore', 'WinnerEloBefore': 'OppEloBefore',
    'L_Set1': 'MySet1', 'L_Set2': 'MySet2', 'L_Set3': 'MySet3',
    'W_Set1': 'OppSet1', 'W_Set2': 'OppSet2', 'W_Set3': 'OppSet3',
}).assign(Won=0)

player_log = pd.concat([winner_rows, loser_rows], ignore_index=True)
player_log['LokacijaNorm'] = player_log['Lokacija'].str.strip().str.lower()
# CRITICAL: strict ascending (IgracId, Datum, SusretId) order -- every point-in-time feature
# below only ever looks at rows ABOVE the current one within a player's own group, i.e.
# strictly BEFORE this match. Same discipline as notebooks 02/05/08/10.
player_log = player_log.sort_values(['IgracId', 'Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
print(f'player_log: {player_log.shape}')
player_log.head(3)


## Step 3 — Per-step point-in-time features (identical to Exp C3)

In [ ]:
g = player_log.groupby('IgracId', sort=False)
player_log['MatchCountBefore'] = g.cumcount()
player_log['HomeCountBefore'] = player_log.groupby(['IgracId', 'LokacijaNorm'], sort=False).cumcount()
player_log['HomeShareBefore'] = np.where(
    player_log['MatchCountBefore'] > 0,
    player_log['HomeCountBefore'] / player_log['MatchCountBefore'],
    np.nan,
)
player_log['DaysSinceLastMatch'] = player_log.groupby('IgracId')['Datum'].diff().dt.days
player_log['CareerWinRateBefore'] = g['Won'].transform(lambda s: s.shift().expanding().mean())

for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']:
    player_log[c + '_f'] = player_log[c].fillna(0)
player_log['SetMargin'] = (
    (player_log['MySet1_f'] + player_log['MySet2_f'] + player_log['MySet3_f'])
    - (player_log['OppSet1_f'] + player_log['OppSet2_f'] + player_log['OppSet3_f'])
)
player_log = player_log.drop(columns=[c + '_f' for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']])

check_w = raw[['SusretId', 'WinnerId', 'WinnerMatchCountBefore']].rename(columns={'WinnerId': 'IgracId', 'WinnerMatchCountBefore': 'Exp'})
check_l = raw[['SusretId', 'LoserId', 'LoserMatchCountBefore']].rename(columns={'LoserId': 'IgracId', 'LoserMatchCountBefore': 'Exp'})
chk = pd.concat([check_w, check_l])
mm = player_log.merge(chk, on=['SusretId', 'IgracId'], how='left')
mismatches = (mm['MatchCountBefore'] != mm['Exp']).sum()
assert mismatches == 0, 'pandas point-in-time ordering does not match the SQL export!'
print(f'MatchCountBefore cross-check vs SQL export: {mismatches} mismatches -- OK.')
player_log[['Won', 'SetMargin', 'DaysSinceLastMatch', 'LigaKoeficjent', 'HomeShareBefore', 'CareerWinRateBefore']].describe()


## Step 4 — Opponent-quality-at-that-time proxy (identical to Exp C3)

In [ ]:
opp_cwr = player_log[['SusretId', 'IgracId', 'CareerWinRateBefore']].rename(
    columns={'IgracId': 'OpponentId', 'CareerWinRateBefore': 'OpponentCareerWinRateBefore'})
player_log = player_log.merge(opp_cwr, on=['SusretId', 'OpponentId'], how='left', validate='many_to_one')
print(f"OpponentCareerWinRateBefore NaN share (opponent's own debut match): "
      f"{player_log['OpponentCareerWinRateBefore'].isna().mean():.2%}")
player_log[['OpponentCareerWinRateBefore']].describe()


## Step 5 — Per-step H2H features (`WasH2H`, `H2HSetsMarginBefore`) — identical to Exp C3

Vectorized (`groupby`/`cumcount`/`cumsum`), strictly point-in-time by the same
`(IgracId, Datum, SusretId)` global-sort argument used throughout this project. Reused unchanged
from `10_enriched_gru.ipynb` — this notebook does not touch the per-step sequence at all, only the
head-level text block (Step 8 below).

### Leakage spot-check (mandatory — reapplied here)

In [ ]:
h2h_g = player_log.groupby(['IgracId', 'OpponentId'], sort=False)
player_log['H2HMeetingsBefore'] = h2h_g.cumcount()          # prior-meeting COUNT, strictly before this row
player_log['WasH2H'] = (player_log['H2HMeetingsBefore'] > 0).astype(np.float32)  # binary flag
player_log['H2HSetsMarginBefore'] = h2h_g['SetMargin'].transform(lambda s: s.cumsum().shift(fill_value=0.0))

sample_pair = player_log.loc[player_log['H2HMeetingsBefore'] >= 2].iloc[0]
pair_hist = player_log[
    (player_log['IgracId'] == sample_pair['IgracId'])
    & (player_log['OpponentId'] == sample_pair['OpponentId'])
    & (player_log['Datum'] < sample_pair['Datum'])
]
assert len(pair_hist) == sample_pair['H2HMeetingsBefore'], 'WasH2H/H2HMeetingsBefore leakage!'
expected_margin = pair_hist['SetMargin'].sum()
assert abs(expected_margin - sample_pair['H2HSetsMarginBefore']) < 1e-6, 'H2HSetsMarginBefore leakage/mismatch!'
print(f"OK -- H2H leakage spot-check: IgracId={sample_pair['IgracId']} vs OpponentId={sample_pair['OpponentId']}: "
      f"{len(pair_hist)} strictly-prior meetings independently recomputed, sets margin "
      f"{expected_margin:+.1f} == {sample_pair['H2HSetsMarginBefore']:+.1f} (column value). No leakage.")
print(f"\nWasH2H positive share (has faced this exact opponent before): {player_log['WasH2H'].mean():.2%}")


## Step 6 — Build the 9-dim per-step feature array + history precomputation (identical to Exp C3)

`MAX_LEN=24`. Per-step dims unchanged: `[Won, SetMarginScaled, DaysRestScaled, LigaKoeficjentScaled,
HomeShareBefore, OpponentCareerWinRateBefore, OppEloBeforeScaled, WasH2H, IsPlaceholder]`. This
notebook's contribution is entirely in the HEAD (Step 8), not the sequence.

In [ ]:
MAX_LEN = 24
N_FEATS = 9  # Won, SetMarginScaled, DaysRestScaled, LigaKoefScaled, HomeShareBefore,
             # OpponentCareerWinRateBefore, OppEloBeforeScaled, WasH2H, IsPlaceholder

SETMARGIN_SCALE = 12.0   # fixed domain constant, unchanged from Exp C/C2/C3
DAYSREST_CLIP = 90.0     # fixed domain constant, unchanged from Exp C/C2/C3

_split_date = pd.Timestamp(TAGS['split_date'])
_train_mask_pl = player_log['Datum'] < _split_date
LIGAKOEF_SCALE = float(player_log.loc[_train_mask_pl, 'LigaKoeficjent'].max())  # train-only max
print(f'LIGAKOEF_SCALE (train-only) = {LIGAKOEF_SCALE}')

# ELO scaling -- TRAIN-ONLY mean/std pooled over player_elo/opponent_elo from `sym`, identical
# source/discipline to Exp C2/C3. Reused for both head-level ELO scalars and per-step OppEloBefore.
train_elo_mask = sym['Split'] == 'train'
ELO_MEAN = float(pd.concat([sym.loc[train_elo_mask, 'player_elo'], sym.loc[train_elo_mask, 'opponent_elo']]).mean())
ELO_STD = float(pd.concat([sym.loc[train_elo_mask, 'player_elo'], sym.loc[train_elo_mask, 'opponent_elo']]).std())
print(f'ELO_MEAN (train-only) = {ELO_MEAN:.2f}, ELO_STD (train-only) = {ELO_STD:.2f}')

step_feats = np.zeros((len(player_log), N_FEATS), dtype=np.float32)
step_feats[:, 0] = player_log['Won'].values
step_feats[:, 1] = np.clip(player_log['SetMargin'].values / SETMARGIN_SCALE, -3, 3)
days_rest = player_log['DaysSinceLastMatch'].fillna(DAYSREST_CLIP).clip(upper=DAYSREST_CLIP).values
step_feats[:, 2] = days_rest / DAYSREST_CLIP
step_feats[:, 3] = player_log['LigaKoeficjent'].values / LIGAKOEF_SCALE
step_feats[:, 4] = player_log['HomeShareBefore'].fillna(0.0).values
step_feats[:, 5] = player_log['OpponentCareerWinRateBefore'].fillna(0.5).values
step_feats[:, 6] = np.clip((player_log['OppEloBefore'].values - ELO_MEAN) / ELO_STD, -4, 4)
step_feats[:, 7] = player_log['WasH2H'].values
step_feats[:, 8] = 0.0  # real step

PLACEHOLDER_VEC = np.zeros(N_FEATS, dtype=np.float32)
PLACEHOLDER_VEC[5] = 0.5   # neutral opponent career win rate
PLACEHOLDER_VEC[6] = 0.0   # neutral (mean) opponent ELO -- 0 after centering
PLACEHOLDER_VEC[7] = 0.0   # no prior H2H possible at a debut match
PLACEHOLDER_VEC[8] = 1.0   # IsPlaceholder flag

player_log['_gpos'] = np.arange(len(player_log))
first_idx = player_log.groupby('IgracId')['_gpos'].min().to_dict()
idx_lookup = dict(zip(zip(player_log['IgracId'], player_log['SusretId']), player_log['_gpos']))

def build_history(igrac_id, susret_id):
    '''(seq[MAX_LEN, N_FEATS] float32, length) -- history STRICTLY before this match.'''
    gpos = idx_lookup[(igrac_id, susret_id)]
    start = first_idx[igrac_id]
    p = gpos - start
    seq = np.zeros((MAX_LEN, N_FEATS), dtype=np.float32)
    if p == 0:
        seq[0] = PLACEHOLDER_VEC
        length = 1
    else:
        lo = max(start, gpos - MAX_LEN)
        chunk = step_feats[lo:gpos]
        length = chunk.shape[0]
        seq[:length] = chunk
    return seq, length

n_rows = len(player_log)
HIST_ARRAY = np.zeros((n_rows, MAX_LEN, N_FEATS), dtype=np.float32)
LEN_ARRAY = np.zeros(n_rows, dtype=np.int64)
igrac_arr, susret_arr = player_log['IgracId'].values, player_log['SusretId'].values
t_hist = time.time()
for i in range(n_rows):
    HIST_ARRAY[i], LEN_ARRAY[i] = build_history(igrac_arr[i], susret_arr[i])
print(f'Precomputed histories for {n_rows:,} player-match rows in {time.time()-t_hist:.2f}s.')
print(f'Sequence length -- min={LEN_ARRAY.min()}, median={np.median(LEN_ARRAY):.0f}, max={LEN_ARRAY.max()}')


### Leakage spot-check (mandatory — reapplied here)

In [ ]:
p_check = (player_log['_gpos'] - player_log['IgracId'].map(first_idx))
assert (p_check == player_log['MatchCountBefore']).all(), 'group contiguity assumption violated!'
print('OK -- p (derived from global sort) == MatchCountBefore for all rows (group contiguity holds).')

test_period_start = pd.Timestamp(TAGS['split_date'])
sample = player_log.loc[player_log['Datum'] >= test_period_start].iloc[100]
gpos, start = sample['_gpos'], first_idx[sample['IgracId']]
hist_rows = player_log.iloc[start:gpos]
print(f"\nSpot check -- IgracId={sample['IgracId']}, SusretId={sample['SusretId']}, "
      f"Datum={sample['Datum'].date()}, MatchCountBefore={sample['MatchCountBefore']}")
print(f'History rows found: {len(hist_rows)} (expected {sample["MatchCountBefore"]})')
assert (hist_rows['Datum'] < sample['Datum']).all(), 'LEAKAGE: a history row is not strictly prior!'
print('OK -- every history row is strictly before the predicted match date. No leakage.')


## Step 7 — Resolve `features.parquet` rows onto history positions + chronological carve (identical to Exp C3)

In [ ]:
sym = sym.sort_values(['Datum', 'SusretId', 'Perspective'], kind='mergesort').reset_index(drop=True)
player_pos = np.array([idx_lookup[(pid, sid)] for pid, sid in zip(sym['PlayerId'], sym['SusretId'])])
opp_pos = np.array([idx_lookup[(oid, sid)] for oid, sid in zip(sym['OpponentId'], sym['SusretId'])])
print(f'Resolved player/opponent history positions for {len(sym):,} rows.')

train_matches = (
    sym.loc[sym['Split'] == 'train', ['SusretId', 'Datum']]
    .drop_duplicates().sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
)
n_train_matches = len(train_matches)
CALIB_FRAC, EARLY_STOP_FRAC = 0.15, 0.10
n_calib = int(round(n_train_matches * CALIB_FRAC))
fit_order = train_matches.iloc[: n_train_matches - n_calib].reset_index(drop=True)
calib_ids = set(train_matches.iloc[n_train_matches - n_calib:]['SusretId'])
n_fit = len(fit_order)
n_eval = int(round(n_fit * EARLY_STOP_FRAC))
core_ids = set(fit_order.iloc[: n_fit - n_eval]['SusretId'])
eval_ids = set(fit_order.iloc[n_fit - n_eval:]['SusretId'])

sym['Bucket'] = np.where(sym['Split'] == 'test', 'test',
                  np.where(sym['SusretId'].isin(calib_ids), 'calib',
                  np.where(sym['SusretId'].isin(eval_ids), 'eval', 'core')))
print(sym['Bucket'].value_counts())
assert sym.loc[sym['Bucket'] == 'test'].shape[0] == TAGS['n_matches_test'] * 2


## Step 8 — Head-level scalar blocks, including the REAL TextSkill join (the change vs Exp C3)

**Per-player (×2), unchanged from Exp C3:** ELO, `CareerWinRateBefore`/`FormaLast10` (NaN→0.5),
`MatchCountBefore` (÷100, clipped), `Age` (TRAIN-only mean/std, NaN→0 after centering).

**Per-player (×2), NEW — real TextSkillScore (Step 8b below):** `text_skill_p`/`text_skill_o`,
`text_conf_p`/`text_conf_o`, `has_text_p`/`has_text_o` — replaces Exp C3's 2 neutral `text_p`/
`text_o` columns with 6 real ones. See the notebook intro above for the full encoding rationale.

**Per-matchup (×1), unchanged from Exp C3:** `h2h_win_rate`, `h2h_total`, `h2h_sets_margin`.

In [ ]:
sym['elo_p_scaled'] = (sym['player_elo'] - ELO_MEAN) / ELO_STD
sym['elo_o_scaled'] = (sym['opponent_elo'] - ELO_MEAN) / ELO_STD

# CareerWinRateBefore / FormaLast10 -- already bounded [0,1], reuse raw, NaN->0.5 neutral.
sym['cwr_p'] = sym['player_career_winrate_before'].fillna(0.5)
sym['cwr_o'] = sym['opponent_career_winrate_before'].fillna(0.5)
sym['forma_p'] = sym['player_forma_last10'].fillna(0.5)
sym['forma_o'] = sym['opponent_forma_last10'].fillna(0.5)

# MatchCountBefore -- fixed domain-constant scale (observed max 300, median 37).
MATCHCOUNT_SCALE = 100.0
sym['mc_p'] = np.clip(sym['player_match_count_before'] / MATCHCOUNT_SCALE, 0, 4)
sym['mc_o'] = np.clip(sym['opponent_match_count_before'] / MATCHCOUNT_SCALE, 0, 4)

# Age -- TRAIN-ONLY mean/std (pooled); NaN (~7.8%, missing Godiste) imputed at the train mean.
train_age_mask = sym['Split'] == 'train'
AGE_MEAN = float(pd.concat([sym.loc[train_age_mask, 'player_age'], sym.loc[train_age_mask, 'opponent_age']]).mean())
AGE_STD = float(pd.concat([sym.loc[train_age_mask, 'player_age'], sym.loc[train_age_mask, 'opponent_age']]).std())
sym['age_p'] = ((sym['player_age'] - AGE_MEAN) / AGE_STD).fillna(0.0)
sym['age_o'] = ((sym['opponent_age'] - AGE_MEAN) / AGE_STD).fillna(0.0)
print(f'AGE_MEAN (train-only) = {AGE_MEAN:.1f}, AGE_STD (train-only) = {AGE_STD:.1f}')

# Per-matchup: h2h_win_rate / h2h_total reused raw+scaled; h2h_sets_margin from Step 5.
H2HTOTAL_SCALE = 10.0
H2H_SETSMARGIN_SCALE = 20.0
sym['h2h_wr'] = sym['player_h2h_win_rate_before']
sym['h2h_total_scaled'] = np.clip(sym['h2h_total_before'] / H2HTOTAL_SCALE, 0, 3)
sym['h2h_sets_margin_scaled'] = np.clip(player_log['H2HSetsMarginBefore'].values[player_pos] / H2H_SETSMARGIN_SCALE, -3, 3)


### Step 8b — NEW: real TextSkillScore join (`trenutci_scores.csv`)

Coverage check: `../data/inscope_player_ids.json` lists 779 in-scope players; `trenutci_scores.csv`
scored 671 of them → **86.1% coverage**, matching the expected ~86%. Vectorized `Series.map`
lookups (no per-row loops) for both perspectives (`PlayerId` and `OpponentId`).

In [ ]:
scores = pd.read_csv(DATA_DIR / 'trenutci_scores.csv')[['IgracId', 'skill_1_10', 'confidence']]
with open(DATA_DIR / 'inscope_player_ids.json', encoding='utf-8') as f:
    inscope_ids = json.load(f)
print(f'trenutci_scores.csv: {scores.shape[0]} scored players out of {len(inscope_ids)} in-scope '
      f'({scores.shape[0] / len(inscope_ids):.1%} coverage).')

# TRAIN-only mean/std for skill_1_10 -- pooled over player+opponent perspectives of TRAIN rows,
# restricted to players who actually HAVE a text score. Never touches TEST rows/players (same
# discipline as ELO_MEAN/AGE_MEAN -- Exp C/C2/C3's established train/test hygiene rule).
train_rows_ids = sym.loc[sym['Split'] == 'train', ['PlayerId', 'OpponentId']]
train_player_ids = set(pd.concat([train_rows_ids['PlayerId'], train_rows_ids['OpponentId']]))
train_scored = scores[scores['IgracId'].isin(train_player_ids)]
TEXT_SKILL_MEAN = float(train_scored['skill_1_10'].mean())
TEXT_SKILL_STD = float(train_scored['skill_1_10'].std())
print(f'TEXT_SKILL_MEAN (train-only, scored players) = {TEXT_SKILL_MEAN:.3f}, '
      f'TEXT_SKILL_STD = {TEXT_SKILL_STD:.3f}  (n={len(train_scored)} scored TRAIN players)')

skill_lookup = scores.set_index('IgracId')['skill_1_10'].to_dict()
conf_lookup = scores.set_index('IgracId')['confidence'].to_dict()

for who, id_col in [('p', 'PlayerId'), ('o', 'OpponentId')]:
    raw_skill = sym[id_col].map(skill_lookup)                       # NaN for unscored players
    has_text = raw_skill.notna().astype(np.float32)                 # explicit missingness flag
    skill_scaled = ((raw_skill - TEXT_SKILL_MEAN) / TEXT_SKILL_STD).fillna(0.0)  # neutral (=train mean) fill
    conf_val = sym[id_col].map(conf_lookup).fillna(0.0)              # 0 reliability when no text at all
    sym[f'text_skill_{who}'] = skill_scaled
    sym[f'text_conf_{who}'] = conf_val
    sym[f'has_text_{who}'] = has_text

print(f"\nhas_text_p coverage over ALL sym rows (train+test, both perspectives): {sym['has_text_p'].mean():.2%}")
print(f"has_text_p coverage over TEST rows only: {sym.loc[sym['Bucket']=='test', 'has_text_p'].mean():.2%}")
assert sym[['text_skill_p', 'text_skill_o', 'text_conf_p', 'text_conf_o', 'has_text_p', 'has_text_o']].isna().sum().sum() == 0, \
    'NaN leaked into the text head block!'
print('OK -- no NaNs in the text head block (missing players fully neutral-filled + flagged).')
sym[['text_skill_p', 'text_conf_p', 'has_text_p']].describe()


In [ ]:
HEAD_COLS = ['elo_p_scaled', 'elo_o_scaled', 'cwr_p', 'cwr_o', 'forma_p', 'forma_o',
             'mc_p', 'mc_o', 'age_p', 'age_o',
             'text_skill_p', 'text_skill_o', 'text_conf_p', 'text_conf_o', 'has_text_p', 'has_text_o',
             'h2h_wr', 'h2h_total_scaled', 'h2h_sets_margin_scaled']
N_HEAD = len(HEAD_COLS)
print(f'N_HEAD scalar features: {N_HEAD} -- {HEAD_COLS}')
print(f'(Exp C3 had 15 head scalars with a 2-col neutral text slot; Exp C4 has {N_HEAD} with 6 real text columns.)')
assert sym[HEAD_COLS].isna().sum().sum() == 0, 'NaN leaked into head scalar block!'
print('OK -- no NaNs in the full head scalar block.')
sym[HEAD_COLS].describe()


## Step 9 — Two-tower shared-weight GRU model + enriched head (Exp C4)

Identical `TwoTowerGRU`-family architecture from Exp C/C2/C3 (`hidden_size=24`, dropout `0.3`,
shared weights across both towers, `BCEWithLogitsLoss`, AdamW weight decay `1e-4`, `MAX_LEN=24`,
early stopping on chronological `eval`-bucket Brier, seed 42). **Only the head's input width
changes:** `3×24=72` (GRU embeddings) `+ 19` (enriched head-scalar block, now with real text)
`= 91`, vs. Exp C3's `87`.

In [ ]:
HIST_T = torch.from_numpy(HIST_ARRAY)
LEN_T = torch.from_numpy(LEN_ARRAY)


class TwoTowerGRUEnriched(nn.Module):
    '''Exp C3 TwoTowerGRU architecture, unchanged, with N_HEAD now including the 6 real
    text-skill columns instead of Exp C3's 2 inert neutral-constant columns.'''

    def __init__(self, input_size=N_FEATS, hidden_size=24, dropout=0.3, mlp_hidden=16, n_head=N_HEAD):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3 + n_head, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len, head_feats):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o, head_feats], dim=1)
        return self.head(combined).squeeze(-1)


def bucket_tensors(bucket_name):
    mask = (sym['Bucket'] == bucket_name).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
    hf = torch.as_tensor(sym.loc[mask, HEAD_COLS].values, dtype=torch.float32)
    return HIST_T[pp], LEN_T[pp], HIST_T[op], LEN_T[op], y, hf

core_pseq, core_plen, core_oseq, core_olen, core_y, core_hf = bucket_tensors('core')
eval_pseq, eval_plen, eval_oseq, eval_olen, eval_y, eval_hf = bucket_tensors('eval')
calib_pseq, calib_plen, calib_oseq, calib_olen, calib_y, calib_hf = bucket_tensors('calib')
test_pseq, test_plen, test_oseq, test_olen, test_y, test_hf = bucket_tensors('test')
print(f'core={core_y.shape[0]:,}  eval={eval_y.shape[0]:,}  calib={calib_y.shape[0]:,}  test={test_y.shape[0]:,}')


## Step 10 — Training loop (identical hyperparameters + early stopping to Exp C/C2/C3)

ONE train (hyperparameters reused, not re-tuned, by design). Manual full-tensor
mini-batches, `BATCH_SIZE=512`, `MAX_EPOCHS=60`, `PATIENCE=8`, early stopping on the chronological
`eval` bucket's Brier score (never TEST, never `calib`).

In [ ]:
HIDDEN_SIZE, DROPOUT, WEIGHT_DECAY, LR = 24, 0.3, 1e-4, 2e-3
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 512, 60, 8

torch.manual_seed(RANDOM_STATE)
model = TwoTowerGRUEnriched(hidden_size=HIDDEN_SIZE, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
n_core = core_y.shape[0]

def eval_bucket(p_seq, p_len, o_seq, o_len, hf, y):
    model.eval()
    with torch.no_grad():
        logit = model(p_seq, p_len, o_seq, o_len, hf)
    return brier_score_loss(y.numpy(), torch.sigmoid(logit).numpy())

best_brier, best_state, epochs_no_improve = np.inf, None, 0
history, rng = [], np.random.RandomState(RANDOM_STATE)

t0 = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    perm = rng.permutation(n_core)
    total_loss, n_batches = 0.0, 0
    for start in range(0, n_core, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        optimizer.zero_grad()
        logit = model(core_pseq[idx], core_plen[idx], core_oseq[idx], core_olen[idx], core_hf[idx])
        loss = criterion(logit, core_y[idx])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    val_brier = eval_bucket(eval_pseq, eval_plen, eval_oseq, eval_olen, eval_hf, eval_y)
    history.append({'epoch': epoch, 'train_loss': total_loss / n_batches, 'val_brier': val_brier})

    improved = val_brier < best_brier - 1e-5
    if improved:
        best_brier, best_state, epochs_no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        epochs_no_improve += 1

    if epoch % 5 == 0 or improved or epoch == 1:
        print(f'epoch {epoch:3d}  train_loss={total_loss/n_batches:.4f}  val_brier={val_brier:.5f}'
              f"{'  <- best' if improved else ''}")
    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (no improvement in {PATIENCE} epochs).')
        break

model.load_state_dict(best_state)
print(f'\nTraining done in {time.time()-t0:.1f}s. Best eval-bucket Brier = {best_brier:.5f}')
history_df = pd.DataFrame(history)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df['epoch'], history_df['train_loss'], label='train loss (BCE, core)', color='#2563eb')
ax2 = ax.twinx()
ax2.plot(history_df['epoch'], history_df['val_brier'], label='val Brier (eval bucket)', color='#dc2626')
ax.set_xlabel('epoch'); ax.set_ylabel('train BCE loss', color='#2563eb'); ax2.set_ylabel('val Brier', color='#dc2626')
ax.set_title('Exp C4 training curve (enriched GRU + real text skill) -- early stopping on chronological eval-bucket Brier')
fig.tight_layout()
plt.savefig('../models/exp_c4_training_curve.png', dpi=110)
plt.show()


## Step 11 — Calibration on the `calib` bucket (isotonic only, by design)

Exp C3 ran a temperature-vs-isotonic bake-off; this notebook is scoped to isotonic regression
only, so it fits isotonic regression on the `calib` slice and applies it
directly to TEST — no bake-off repeated here.

In [ ]:
model.eval()
with torch.no_grad():
    calib_logit = model(calib_pseq, calib_plen, calib_oseq, calib_olen, calib_hf)
    test_logit = model(test_pseq, test_plen, test_oseq, test_olen, test_hf)

calib_logit_np, calib_y_np = calib_logit.numpy(), calib_y.numpy()
test_logit_np, test_y_np = test_logit.numpy(), test_y.numpy()

p_calib_raw = 1 / (1 + np.exp(-calib_logit_np))
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(p_calib_raw, calib_y_np)
p_calib_iso = iso.predict(p_calib_raw)

brier_raw = brier_score_loss(calib_y_np, p_calib_raw)
brier_iso = brier_score_loss(calib_y_np, p_calib_iso)
print(f'Calib-slice Brier -- raw: {brier_raw:.5f}, isotonic: {brier_iso:.5f}')

calib_method = 'isotonic'
p_test_raw = 1 / (1 + np.exp(-test_logit_np))
p_test_calibrated = iso.predict(p_test_raw)
print(f'Calibration method for TEST: {calib_method} (isotonic, by design)')


## Step 12 — Score TEST (identical 6,852-row set) — full comparison Exp 0/A/A'/B/B'/C/C2/C3/C4

In [ ]:
def compute_metrics(y_true, p, label):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        'experiment': label,
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
        'auc': roc_auc_score(y_true, p),
    }

preds4 = pd.read_parquet(DATA_DIR / 'test_predictions_4way.parquet')
preds_traj = pd.read_parquet(DATA_DIR / 'test_predictions_trajectory.parquet')
preds_c2 = pd.read_parquet(DATA_DIR / 'test_predictions_c2.parquet')
preds_c3 = pd.read_parquet(DATA_DIR / 'test_predictions_c3.parquet')

test_bucket_keys = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'PlayerId', 'OpponentId', 'Label']]
merge_check = test_bucket_keys.merge(preds4[['SusretId', 'PlayerId', 'OpponentId', 'Label']],
                                      on=['SusretId', 'PlayerId', 'OpponentId'], suffixes=('_c4', '_exp'))
assert len(merge_check) == len(test_bucket_keys) == len(preds4), 'Row-set mismatch vs Exp 0/A/B/C test set!'
assert (merge_check['Label_c4'] == merge_check['Label_exp']).all(), 'Label mismatch vs Exp 0/A/B/C test set!'
print(f'OK -- Exp C4 test bucket is the exact same {len(preds4):,}-row set as every prior experiment, labels match.')

elo_brier = brier_score_loss(preds4['Label'], preds4['p_exp0_elo'])
comparison = pd.DataFrame([
    compute_metrics(preds4['Label'].values, preds4['p_exp0_elo'].values, 'Exp 0 - ELO baseline'),
    compute_metrics(preds4['Label'].values, preds4['p_expA_calibrated'].values, 'Exp A - ELO-free XGBoost'),
    compute_metrics(preds_traj['Label'].values, preds_traj['p_expAprime_calibrated'].values, "Exp A' - ELO-free XGB + trajectory"),
    compute_metrics(preds4['Label'].values, preds4['p_expB_calibrated'].values, 'Exp B - ELO-included XGBoost'),
    compute_metrics(preds_traj['Label'].values, preds_traj['p_expBprime_calibrated'].values, "Exp B' - ELO-included XGB + trajectory"),
    compute_metrics(preds4['Label'].values, preds4['p_expC_calibrated'].values, 'Exp C - ELO-free GRU'),
    compute_metrics(preds_c2['Label'].values, preds_c2['p_expC2_calibrated'].values, 'Exp C2 - GRU + ELO'),
    compute_metrics(preds_c3['Label'].values, preds_c3['p_expC3_calibrated'].values, 'Exp C3 - enriched GRU (no text)'),
    compute_metrics(test_y_np, p_test_calibrated, 'Exp C4 - enriched GRU + text skill (this notebook)'),
]).set_index('experiment')
comparison['brier_delta_vs_elo_pct'] = (comparison['brier'] - elo_brier) / elo_brier * 100
comparison_display = comparison.round({'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})
comparison_display


In [ ]:
# Headline point-estimate subset requested by the task: Exp 0 / B' / C / C2 / C3 / C4
headline_subset = comparison_display.loc[[
    'Exp 0 - ELO baseline', "Exp B' - ELO-included XGB + trajectory",
    'Exp C - ELO-free GRU', 'Exp C2 - GRU + ELO', 'Exp C3 - enriched GRU (no text)',
    'Exp C4 - enriched GRU + text skill (this notebook)',
]]
print("Headline subset (Exp 0 / B' / C / C2 / C3 / C4):")
headline_subset


In [ ]:
print('Headline point-estimate verdict:')
for exp in comparison.index:
    d = comparison.loc[exp, 'brier_delta_vs_elo_pct']
    print(f"  {exp}: Brier {d:+.2f}% vs ELO baseline ({'WORSE' if d > 0 else 'better'} -- higher Brier = worse).")

raw_metrics = compute_metrics(test_y_np, p_test_raw, 'Exp C4 - raw (uncalibrated)')
cal_metrics = compute_metrics(test_y_np, p_test_calibrated, 'Exp C4 - calibrated')
pd.DataFrame([raw_metrics, cal_metrics]).set_index('experiment').round(5)


## Step 13 — Match-level bootstrap CIs + paired-bootstrap Brier differences (full TEST set)

Identical methodology to `06`/`08`/`10`: 2,000 resamples, seed 42, both perspective rows of a
match always travel together (drawing match IDs with replacement, not individual rows).

In [ ]:
preds_all = preds4.merge(preds_traj[['SusretId', 'PlayerId', 'OpponentId', 'p_expAprime_calibrated', 'p_expBprime_calibrated']],
                          on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
preds_all = preds_all.merge(preds_c2[['SusretId', 'PlayerId', 'OpponentId', 'p_expC2_calibrated']],
                             on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
preds_all = preds_all.merge(preds_c3[['SusretId', 'PlayerId', 'OpponentId', 'p_expC3_calibrated']],
                             on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
exp_c4_preds = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'PlayerId', 'OpponentId']].copy()
exp_c4_preds['p_expC4_calibrated'] = p_test_calibrated
merged = preds_all.merge(exp_c4_preds, on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
assert len(merged) == len(preds4) == len(preds_traj) == len(preds_c2) == len(preds_c3), 'row-set mismatch across prediction files!'
print(f'Merged all 9 experiments onto {len(merged):,} identical test rows.')

EXPERIMENTS = {
    'Exp 0 - ELO baseline':                   'p_exp0_elo',
    'Exp A - ELO-free XGB':                   'p_expA_calibrated',
    "Exp A' - ELO-free XGB + trajectory":     'p_expAprime_calibrated',
    'Exp B - ELO-included XGB':               'p_expB_calibrated',
    "Exp B' - ELO-included XGB + trajectory": 'p_expBprime_calibrated',
    'Exp C - ELO-free GRU':                   'p_expC_calibrated',
    'Exp C2 - GRU + ELO':                     'p_expC2_calibrated',
    'Exp C3 - enriched GRU (no text)':        'p_expC3_calibrated',
    'Exp C4 - enriched GRU + text skill':     'p_expC4_calibrated',
}
y_all = merged['Label'].values
p_all = {name: merged[col].values for name, col in EXPERIMENTS.items()}

def m_dict(y_true, p):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    out = {'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
           'brier': brier_score_loss(y_true, p), 'log_loss': log_loss(y_true, p_c)}
    try:
        out['auc'] = roc_auc_score(y_true, p)
    except ValueError:
        out['auc'] = np.nan
    return out

point_estimates = {name: m_dict(y_all, p_all[name]) for name in EXPERIMENTS}

groups = merged.groupby('SusretId').indices
match_ids = np.array(list(groups.keys()))
lens = np.array([len(groups[m]) for m in match_ids])
assert (lens == 2).all(), 'expected exactly 2 rows (both perspectives) per match'
row_idx_2d = np.array([groups[m] for m in match_ids], dtype=np.int64)
n_matches = len(match_ids)
print(f'TEST set: {len(merged):,} rows, {n_matches:,} unique matches.')


In [ ]:
N_BOOT = 2000
rng = np.random.RandomState(RANDOM_STATE)

boot_metrics = {name: {'accuracy': [], 'brier': [], 'log_loss': [], 'auc': []} for name in EXPERIMENTS}
DIFF_KEYS = {
    'C4_minus_C3': ('Exp C4 - enriched GRU + text skill', 'Exp C3 - enriched GRU (no text)'),
    'C4_minus_C2': ('Exp C4 - enriched GRU + text skill', 'Exp C2 - GRU + ELO'),
    'C4_minus_Bprime': ('Exp C4 - enriched GRU + text skill', "Exp B' - ELO-included XGB + trajectory"),
    'C4_minus_0': ('Exp C4 - enriched GRU + text skill', 'Exp 0 - ELO baseline'),
}
boot_diff = {k: [] for k in DIFF_KEYS}

t0 = time.time()
for b in range(N_BOOT):
    sampled_match_pos = rng.randint(0, n_matches, size=n_matches)
    idx = row_idx_2d[sampled_match_pos].reshape(-1)
    y_b = y_all[idx]

    briers = {}
    for name in EXPERIMENTS:
        p_b = p_all[name][idx]
        m = m_dict(y_b, p_b)
        for k in boot_metrics[name]:
            boot_metrics[name][k].append(m[k])
        briers[name] = m['brier']

    for dk, (a, b_) in DIFF_KEYS.items():
        boot_diff[dk].append(briers[a] - briers[b_])

print(f'Match-level bootstrap done: {N_BOOT:,} resamples in {time.time()-t0:.1f}s (seed={RANDOM_STATE})')


### 95% Bootstrap CI table (all 9 experiments)

In [ ]:
rows = []
for name in EXPERIMENTS:
    row = {'experiment': name}
    for metric in ['accuracy', 'brier', 'log_loss', 'auc']:
        arr = np.array(boot_metrics[name][metric])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        row[f'{metric}_point'] = point_estimates[name][metric]
        row[f'{metric}_ci_lo'] = lo
        row[f'{metric}_ci_hi'] = hi
    rows.append(row)

ci_table = pd.DataFrame(rows).set_index('experiment')
ci_table_display = ci_table.round(5)
ci_table_display


### Paired bootstrap on Brier DIFFERENCES (significance) — headline: (C4−C3)

In [ ]:
diff_labels = {
    'C4_minus_C3': "Exp C4 (real text) minus Exp C3 (neutral text) -- THE headline question: does the real text-skill prior help, overall?",
    'C4_minus_C2': 'Exp C4 minus Exp C2 (GRU+ELO, no enrichments at all) -- context',
    'C4_minus_Bprime': "Exp C4 minus Exp B' (ELO-included XGB + trajectory) -- best GRU vs best-of-everything XGB",
    'C4_minus_0': 'Exp C4 minus Exp 0 (ELO baseline) -- vs plain ELO',
}
diff_rows = []
for key, label in diff_labels.items():
    arr = np.array(boot_diff[key])
    lo, hi = np.percentile(arr, [2.5, 97.5])
    significant = not (lo < 0 < hi)
    direction = 'lower Brier (BETTER)' if arr.mean() < 0 else 'higher Brier (WORSE)'
    diff_rows.append({'comparison': key, 'description': label, 'mean_diff': arr.mean(),
                       'ci_lo': lo, 'ci_hi': hi, 'significant_95pct': significant, 'direction_of_mean': direction})

diff_table = pd.DataFrame(diff_rows).set_index('comparison')
diff_table_display = diff_table.copy()
for c in ['mean_diff', 'ci_lo', 'ci_hi']:
    diff_table_display[c] = diff_table_display[c].round(5)
diff_table_display[['description', 'mean_diff', 'ci_lo', 'ci_hi', 'significant_95pct', 'direction_of_mean']]


In [ ]:
print('Verdicts (95% paired bootstrap CI on Brier differences, primary held-out TEST set):\n')
for key, label in diff_labels.items():
    r = diff_table.loc[key]
    verdict = 'YES -- statistically significant' if r['significant_95pct'] else 'NOT statistically significant (CI includes 0)'
    print(f"{label}")
    print(f"  mean diff = {r['mean_diff']:+.5f}   95% CI = [{r['ci_lo']:+.5f}, {r['ci_hi']:+.5f}]   -> {verdict}\n")


## Step 14 — Overlaid calibration curves, Exp 0 / B' / C2 / C3 / C4

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
curves = [
    ('p_exp0_elo', 'Exp 0 - ELO formula', dict(marker='s', linestyle='--', color='#374151')),
    ("p_expBprime_calibrated", "Exp B' - ELO-incl XGB + trajectory", dict(marker='v', linestyle='-', color='#059669')),
    ('p_expC2_calibrated', 'Exp C2 - GRU + ELO', dict(marker='o', linestyle='-', color='#dc2626')),
    ('p_expC3_calibrated', 'Exp C3 - enriched GRU (no text)', dict(marker='D', linestyle='-', color='#7c3aed')),
]
for col, label, style in curves:
    frac_pos, mean_pred = calibration_curve(merged['Label'].values, merged[col].values, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, frac_pos, label=label, **style)

frac_pos_c4, mean_pred_c4 = calibration_curve(merged['Label'].values, merged['p_expC4_calibrated'].values, n_bins=10, strategy='uniform')
ax.plot(mean_pred_c4, frac_pos_c4, label='Exp C4 - enriched GRU + text skill', marker='*', linestyle='-', color='#ea580c', linewidth=2, markersize=10)

ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed win rate')
ax.set_title("Calibration (Reliability Diagram) -- Exp 0 / B' / C2 / C3 / C4, TEST set")
ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../models/calibration_curve_c4.png', dpi=120)
plt.show()


## Step 15 — Permutation importance: where does `text_skill` rank now that it's real?

Same methodology as Exp C3 Step 15 (shuffle one column across TEST rows, reapply the SAME fitted
isotonic calibration — no refitting — measure TEST Brier degradation). In Exp C3, `text_p`/`text_o`
were exactly `0` importance (inert constants, sanity-check only). Here they are real signals.

In [ ]:
STEP_FEAT_NAMES = ['Won', 'SetMarginScaled', 'DaysRestScaled', 'LigaKoefScaled', 'HomeShareBefore',
                    'OpponentCareerWinRateBefore', 'OppEloBeforeScaled', 'WasH2H']  # excludes IsPlaceholder

def _apply_calibration(logit):
    p_raw = torch.sigmoid(logit).numpy()
    return iso.predict(p_raw)  # isotonic only, by design

baseline_brier = brier_score_loss(test_y_np, p_test_calibrated)

def permutation_importance_step(feat_idx, n_repeats=3, seed=42):
    rng_local = np.random.RandomState(seed)
    degradations = []
    for _ in range(n_repeats):
        perm_idx = rng_local.permutation(test_pseq.shape[0])
        p_seq_perm, o_seq_perm = test_pseq.clone(), test_oseq.clone()
        p_seq_perm[:, :, feat_idx] = test_pseq[perm_idx, :, feat_idx]
        o_seq_perm[:, :, feat_idx] = test_oseq[perm_idx, :, feat_idx]
        with torch.no_grad():
            logit_perm = model(p_seq_perm, test_plen, o_seq_perm, test_olen, test_hf)
        p_perm = _apply_calibration(logit_perm)
        degradations.append(brier_score_loss(test_y_np, p_perm) - baseline_brier)
    return np.mean(degradations)

def permutation_importance_head(col_idx, n_repeats=3, seed=42):
    rng_local = np.random.RandomState(seed)
    degradations = []
    for _ in range(n_repeats):
        perm_idx = rng_local.permutation(test_hf.shape[0])
        hf_perm = test_hf.clone()
        hf_perm[:, col_idx] = test_hf[perm_idx, col_idx]
        with torch.no_grad():
            logit_perm = model(test_pseq, test_plen, test_oseq, test_olen, hf_perm)
        p_perm = _apply_calibration(logit_perm)
        degradations.append(brier_score_loss(test_y_np, p_perm) - baseline_brier)
    return np.mean(degradations)

step_importance = pd.Series({name: permutation_importance_step(i) for i, name in enumerate(STEP_FEAT_NAMES)})
head_importance = pd.Series({name: permutation_importance_head(i) for i, name in enumerate(HEAD_COLS)})
perm_importance = pd.concat([step_importance, head_importance]).sort_values(ascending=False)
print('Permutation importance (Brier degradation when shuffled -- higher = more important):')
print(perm_importance.to_string())

TEXT_COLS = {'text_skill_p', 'text_skill_o', 'text_conf_p', 'text_conf_o', 'has_text_p', 'has_text_o'}
fig, ax = plt.subplots(figsize=(8, 7))
perm_importance.plot.barh(ax=ax, color=['#ea580c' if n in TEXT_COLS else '#7c3aed' for n in perm_importance.index])
ax.invert_yaxis()
ax.set_xlabel('Test Brier degradation when shuffled')
ax.set_title('Exp C4 -- per-step + head feature importance (permutation; orange = text-skill block)')
plt.tight_layout()
plt.savefig('../models/exp_c4_feature_importance.png', dpi=120)
plt.show()

print(f"\ntext_skill rank (of {len(perm_importance)} features): "
      f"text_skill_p = #{list(perm_importance.index).index('text_skill_p')+1}, "
      f"text_skill_o = #{list(perm_importance.index).index('text_skill_o')+1}")


## Step 16 — ⭐ Cold-start analysis: does the text prior help low-history players specifically?

**The key hypothesis this notebook was built to test.** Text is a skill PRIOR that should help
MOST when match history is thin (ELO and CareerWinRate are least informative early in a player's
career). Test-set rows are split by **that row's own player's `player_match_count_before`** into
3 buckets (a sensible split on this dataset's distribution — median 66, and enough rows per bucket
for a meaningful bootstrap): `very_low (<10)`, `low (10-30)`, `established (>30)`.

**Bootstrap note:** unlike Step 13's match-level bootstrap (which keeps both perspectives of a
match together), this bucket analysis resamples at the **ROW level within each bucket** — a
match's two perspectives can fall into *different* cold-start buckets (each player has their own,
independent `MatchCountBefore`), so bucket membership is inherently a per-perspective-row property,
not a per-match one.

In [ ]:
test_keys = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'PlayerId', 'OpponentId', 'player_match_count_before']].reset_index(drop=True)
test_keys['p_expC4_calibrated'] = p_test_calibrated

cold = test_keys.merge(
    merged[['SusretId', 'PlayerId', 'OpponentId', 'Label', 'p_exp0_elo', 'p_expC2_calibrated', 'p_expC3_calibrated']],
    on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
assert len(cold) == len(test_keys), 'row-set mismatch merging C4 predictions onto the 9-way merged frame!'

cold['bucket'] = pd.cut(cold['player_match_count_before'], bins=[-1, 9, 30, 1e9],
                         labels=['very_low(<10)', 'low(10-30)', 'established(>30)'])
print('Cold-start bucket sizes (TEST rows):')
print(cold['bucket'].value_counts().reindex(['very_low(<10)', 'low(10-30)', 'established(>30)']))


In [ ]:
COLD_MODELS = {'ELO': 'p_exp0_elo', 'C2': 'p_expC2_calibrated', 'C3': 'p_expC3_calibrated', 'C4': 'p_expC4_calibrated'}
BUCKET_ORDER = ['very_low(<10)', 'low(10-30)', 'established(>30)']

bucket_rows = []
rng_cold = np.random.RandomState(RANDOM_STATE)
N_BOOT_COLD = 2000

for bname in BUCKET_ORDER:
    sub = cold[cold['bucket'] == bname].reset_index(drop=True)
    n_sub = len(sub)
    y_sub = sub['Label'].values
    row = {'bucket': bname, 'n_rows': n_sub}
    for m, col in COLD_MODELS.items():
        p = sub[col].values
        row[f'{m}_brier'] = brier_score_loss(y_sub, p)
        row[f'{m}_accuracy'] = accuracy_score(y_sub, (p >= 0.5).astype(int))

    # row-level paired bootstrap, C4 - C3, WITHIN this bucket (2000 resamples, seed 42)
    p_c3, p_c4 = sub['p_expC3_calibrated'].values, sub['p_expC4_calibrated'].values
    diffs = np.empty(N_BOOT_COLD)
    for b in range(N_BOOT_COLD):
        idx = rng_cold.randint(0, n_sub, size=n_sub)
        diffs[b] = brier_score_loss(y_sub[idx], p_c4[idx]) - brier_score_loss(y_sub[idx], p_c3[idx])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    row['C4_minus_C3_mean'] = diffs.mean()
    row['C4_minus_C3_ci_lo'] = lo
    row['C4_minus_C3_ci_hi'] = hi
    row['C4_minus_C3_significant'] = not (lo < 0 < hi)
    bucket_rows.append(row)

bucket_table = pd.DataFrame(bucket_rows).set_index('bucket').loc[BUCKET_ORDER]
bucket_table_display = bucket_table.round(5)
bucket_table_display


In [ ]:
print('Cold-start verdict (row-level paired bootstrap, C4 - C3 Brier difference, 2,000 resamples, seed 42):\n')
for bname in BUCKET_ORDER:
    r = bucket_table.loc[bname]
    verdict = 'YES -- significant' if r['C4_minus_C3_significant'] else 'NOT significant (CI includes 0)'
    direction = 'text HELPS (lower Brier)' if r['C4_minus_C3_mean'] < 0 else 'text HURTS (higher Brier)'
    print(f"{bname}  (n={int(r['n_rows'])}):")
    print(f"  C3 Brier={r['C3_brier']:.5f}  C4 Brier={r['C4_brier']:.5f}  "
          f"mean diff={r['C4_minus_C3_mean']:+.5f}  95% CI=[{r['C4_minus_C3_ci_lo']:+.5f}, {r['C4_minus_C3_ci_hi']:+.5f}]")
    print(f"  -> {direction}, {verdict}\n")


## Step 17 — Retrain-based ablation: `FULL` (Exp C4) vs `NO_TEXT` (zero the 6 text head columns)

A controlled, same-pipeline confirmation of Step 15/16's findings — not a re-tuned model, a pure
input-masking ablation (same discipline as `05`/`10` Step 15/16): retrain the identical
architecture/hyperparameters/seed/splits with the 6 text head columns zeroed out, compare test
Brier to the already-trained `FULL` (Exp C4) model. This isolates the text block's marginal
contribution within the EXACT SAME notebook/pipeline (more apples-to-apples than the cross-notebook
Exp C3 vs Exp C4 comparison above, which uses two separately-trained models).

In [ ]:
TEXT_HEAD_IDX = [HEAD_COLS.index(c) for c in ['text_skill_p', 'text_skill_o', 'text_conf_p', 'text_conf_o', 'has_text_p', 'has_text_o']]

def run_no_text_variant(seed=RANDOM_STATE, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    hf_variant = sym[HEAD_COLS].values.copy()
    hf_variant[:, TEXT_HEAD_IDX] = 0.0

    def _bt(bucket_name):
        mask = (sym['Bucket'] == bucket_name).values
        pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
        op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
        y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
        hf = torch.as_tensor(hf_variant[mask], dtype=torch.float32)
        return HIST_T[pp], LEN_T[pp], HIST_T[op], LEN_T[op], y, hf

    v_core, v_eval, v_calib, v_test = _bt('core'), _bt('eval'), _bt('calib'), _bt('test')

    torch.manual_seed(seed)
    m = TwoTowerGRUEnriched(hidden_size=HIDDEN_SIZE, dropout=DROPOUT).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    crit = nn.BCEWithLogitsLoss()
    n_v_core = v_core[4].shape[0]
    rng_local = np.random.RandomState(seed)

    best_brier_v, best_state_v, no_improve = np.inf, None, 0
    for epoch in range(1, max_epochs + 1):
        m.train()
        perm = rng_local.permutation(n_v_core)
        for start in range(0, n_v_core, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            opt.zero_grad()
            logit = m(v_core[0][idx], v_core[1][idx], v_core[2][idx], v_core[3][idx], v_core[5][idx])
            loss = crit(logit, v_core[4][idx])
            loss.backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            val_logit = m(v_eval[0], v_eval[1], v_eval[2], v_eval[3], v_eval[5])
        val_brier = brier_score_loss(v_eval[4].numpy(), torch.sigmoid(val_logit).numpy())
        if val_brier < best_brier_v - 1e-5:
            best_brier_v, best_state_v, no_improve = val_brier, {k: v.clone() for k, v in m.state_dict().items()}, 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    m.load_state_dict(best_state_v)

    m.eval()
    with torch.no_grad():
        v_calib_logit = m(v_calib[0], v_calib[1], v_calib[2], v_calib[3], v_calib[5])
        v_test_logit = m(v_test[0], v_test[1], v_test[2], v_test[3], v_test[5])
    vc_logit_np, vc_y_np = v_calib_logit.numpy(), v_calib[4].numpy()
    vt_logit_np, vt_y_np = v_test_logit.numpy(), v_test[4].numpy()

    p_vc_raw = 1 / (1 + np.exp(-vc_logit_np))
    iso_v = IsotonicRegression(out_of_bounds='clip')
    iso_v.fit(p_vc_raw, vc_y_np)
    p_vt_final = iso_v.predict(1 / (1 + np.exp(-vt_logit_np)))

    row = compute_metrics(vt_y_np, p_vt_final, 'NO_TEXT')
    return row, best_brier_v

t_abl = time.time()
row_full = compute_metrics(test_y_np, p_test_calibrated, 'FULL (Exp C4)')
row_full['brier_delta_vs_elo_pct'] = (row_full['brier'] - elo_brier) / elo_brier * 100
row_no_text, best_brier_no_text = run_no_text_variant()
row_no_text['brier_delta_vs_elo_pct'] = (row_no_text['brier'] - elo_brier) / elo_brier * 100
print(f"NO_TEXT variant retrained in {time.time()-t_abl:.1f}s (best eval Brier={best_brier_no_text:.5f})")

ablation_table = pd.DataFrame([row_full, row_no_text]).set_index('experiment')
ablation_table_display = ablation_table.round({'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})
ablation_table_display['brier_delta_vs_full'] = (ablation_table['brier'] - ablation_table.loc['FULL (Exp C4)', 'brier']).round(5)
ablation_table_display


## Step 18 — Save artifacts

In [ ]:
torch.save({
    'model_state_dict': best_state,
    'hyperparams': {
        'hidden_size': HIDDEN_SIZE, 'dropout': DROPOUT, 'weight_decay': WEIGHT_DECAY,
        'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN, 'n_feats': N_FEATS,
        'n_head': N_HEAD, 'random_state': RANDOM_STATE,
    },
    'scaling': {
        'setmargin_scale': SETMARGIN_SCALE, 'daysrest_clip': DAYSREST_CLIP, 'ligakoef_scale': LIGAKOEF_SCALE,
        'elo_mean': ELO_MEAN, 'elo_std': ELO_STD, 'age_mean': AGE_MEAN, 'age_std': AGE_STD,
        'matchcount_scale': MATCHCOUNT_SCALE, 'h2htotal_scale': H2HTOTAL_SCALE,
        'h2h_setsmargin_scale': H2H_SETSMARGIN_SCALE,
        'text_skill_mean': TEXT_SKILL_MEAN, 'text_skill_std': TEXT_SKILL_STD,
    },
    'step_feature_names': STEP_FEAT_NAMES + ['IsPlaceholder'],
    'head_feature_names': HEAD_COLS,
    'text_skill_slot_note': 'text_skill_p/o, text_conf_p/o, has_text_p/o now REAL (from trenutci_scores.csv), replacing Exp C3\'s 2 neutral text_p/text_o columns.',
    'calibration': {'method': calib_method},
    'best_eval_brier': float(best_brier),
    'test_metrics': compute_metrics(test_y_np, p_test_calibrated, 'Exp C4 - enriched GRU + text skill (calibrated)'),
}, MODELS_DIR / 'exp_c4_gru_text.pt')

with open(MODELS_DIR / 'exp_c4_isotonic.pkl', 'wb') as f:
    pickle.dump(iso, f)

exp_c4_preds_full = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'PlayerId', 'OpponentId', 'Label', 'Perspective']].copy()
exp_c4_preds_full['p_expC4_raw'] = p_test_raw
exp_c4_preds_full['p_expC4_calibrated'] = p_test_calibrated
exp_c4_preds_full.to_parquet(DATA_DIR / 'test_predictions_c4.parquet', index=False)
merged.to_parquet(DATA_DIR / 'test_predictions_9way.parquet', index=False)
cold.to_parquet(DATA_DIR / 'test_predictions_coldstart.parquet', index=False)

print(f"Saved: {MODELS_DIR / 'exp_c4_gru_text.pt'}")
print(f"Saved: {MODELS_DIR / 'exp_c4_isotonic.pkl'}")
print(f"Saved: {DATA_DIR / 'test_predictions_c4.parquet'}")
print(f"Saved: {DATA_DIR / 'test_predictions_9way.parquet'}")
print(f"Saved: {DATA_DIR / 'test_predictions_coldstart.parquet'}")


## Step 19 — Written Verdict

**Comparison table (calibrated test Brier, held-out 6,852-row / 3,426-match TEST set, 2025-01-19..2026-07-08):**

| Experiment | Brier | Accuracy | AUC | vs ELO |
|---|---|---|---|---|
| Exp 0 -- ELO baseline | 0.17713 | 0.7518 | 0.8210 | -- |
| Exp A' -- ELO-free XGB + trajectory | 0.17542 | 0.7344 | 0.8172 | -0.97% |
| Exp B' -- ELO-included XGB + trajectory | 0.16545 | 0.7561 | 0.8394 | -6.60% |
| Exp C -- ELO-free GRU | 0.16278 | 0.7621 | 0.8411 | -8.11% |
| Exp C2 -- GRU + ELO | 0.16088 | 0.7602 | 0.8455 | -9.18% |
| **Exp C3 -- enriched GRU (no text)** | **0.15869** | **0.7644** | **0.8502** | **-10.41%** |
| Exp C4 -- enriched GRU + real text skill (this notebook) | 0.15982 | 0.7608 | 0.8485 | -9.78% |

**Exp C3 (no text) remains the best model in the project.** Plugging in the real `skill_1_10`
text-skill score (+ confidence + has_text) in place of Exp C3's neutral constant made the point
estimate very slightly *worse* -- Brier +0.00113 (+0.7% relative), accuracy -0.4pp, AUC -0.0017 --
not a rounding artifact, but a small, consistent degradation across every point-estimate metric.

**Paired bootstrap verdicts (2,000 match-level resamples, seed 42, primary held-out TEST set):**

| Comparison | Mean Delta Brier | 95% CI | Verdict |
|---|---|---|---|
| **C4 - C3 (headline: does real text help, overall?)** | **+0.00113** | **[-0.00023, +0.00247]** | **NOT significant** (CI includes 0, and centered on the *worse* side) |
| C4 - C2 (vs bare GRU+ELO, no enrichments at all) | -0.00108 | [-0.00382, +0.00159] | NOT significant |
| C4 - B' (best GRU vs best-of-everything XGB) | -0.00561 | [-0.00941, -0.00184] | Significant (C4 still beats XGB) |
| C4 - 0 (vs plain ELO) | -0.01736 | [-0.02160, -0.01302] | Significant (C4 still crushes plain ELO) |

**Headline verdict: adding the real text-skill prior does NOT help overall, and the 95% CI is not
even centered near zero -- the point estimate moved in the WRONG direction.** This is not "no
effect", it's "a small negative point estimate that bootstrap cannot rule out is noise, but also
cannot spin as a positive result." Exp C4 remains a strong model in absolute terms (still
significantly better than Exp B' and plain ELO), but the specific question this notebook exists to
answer -- does the LLM-derived text prior improve on Exp C3 -- resolves to **no, not on this test
set, and if anything, a small negative point estimate.**

**Cold-start analysis (the key hypothesis -- row-level paired bootstrap within each bucket, 2,000
resamples, seed 42):**

| Bucket (prior matches) | n rows | ELO Brier | C2 Brier | C3 Brier | C4 Brier | C4-C3 mean | 95% CI | Significant? |
|---|---|---|---|---|---|---|---|---|
| very_low (<10) | 1,076 | 0.21193 | 0.18529 | 0.18135 | 0.18385 | +0.00253 | [-0.00071, +0.00566] | No |
| low (10-30) | 1,185 | 0.17648 | 0.16017 | 0.15963 | 0.16029 | +0.00069 | [-0.00210, +0.00351] | No |
| established (>30) | 4,591 | 0.16915 | 0.15534 | 0.15314 | 0.15406 | +0.00092 | [-0.00023, +0.00213] | No |

**The cold-start hypothesis is NOT confirmed -- if anything, the opposite pattern shows up in the
point estimates.** The text prior does not help the very-low-history bucket more than the
established bucket; the point-estimate degradation from adding text is actually *largest* in the
very-low bucket (+0.00253, the biggest of the three) and smallest in the low bucket (+0.00069).
None of the three per-bucket differences are statistically significant (all three 95% CIs include
0), so the honest reading is "no detectable effect in either direction, in any bucket" -- but the
directional story a skill-prior hypothesis would predict (bigger help where ELO/career stats are
thinnest) simply is not there in the point estimates. `has_text` coverage is also fairly even
across buckets (roughly 83-87% coverage on TEST, driven by which players happened to fill in
`Trenutci`, not by experience level), so this isn't an artifact of cold-start players being
systematically less likely to have a text score.

**Retrain-based ablation -- same architecture/pipeline, text head columns zeroed (a controlled,
same-notebook confirmation, not just the cross-notebook C3-vs-C4 comparison above):**

| Variant | Test Brier | Delta vs FULL |
|---|---|---|
| FULL (Exp C4, with real text) | 0.15982 | -- |
| NO_TEXT (6 text head columns zeroed, retrained) | 0.15907 | **-0.00075 (slightly better without text)** |

This retrain-based ablation -- built entirely within this notebook's own pipeline, controlling for
any subtle differences between the separately-trained Exp C3 and Exp C4 models -- agrees with the
cross-notebook comparison: removing the text block is a small net *improvement*, not a loss. Three
independent analyses (cross-notebook C3 vs C4, cold-start buckets, and this same-pipeline ablation)
all point the same direction.

**Permutation importance -- `text_skill` rank now that it's real (vs the inert 0 in Exp C3):**

| Feature | Brier degradation when shuffled | Rank (of 27) |
|---|---|---|
| `elo_p_scaled` / `elo_o_scaled` | 0.0939 / 0.0896 | #1 / #2 (still dominant, as in C3) |
| `SetMarginScaled` | 0.0487 | #3 |
| `OppEloBeforeScaled` | 0.0166 | #4 |
| ... | ... | ... |
| **`text_skill_p`** | **0.00038** | **#17** |
| **`text_skill_o`** | **0.00033** | **#18** |
| `has_text_p` / `text_conf_o` / `text_conf_p` / `has_text_o` | 0.00009 - 0.00017 | #22-#26 |
| `WasH2H` / `h2h_wr` | slightly negative | #26-#27 (bottom, as in C3) |

The model DOES use `text_skill` now that it's real -- it moved from "exactly 0, confirmed inert" in
Exp C3 to a small but genuinely positive permutation importance in Exp C4 (shuffling it measurably
hurts TEST Brier), ranking #17-18 out of 27 features -- above `WasH2H`, `h2h_wr`, and `HomeShareBefore`,
but well below ELO, SetMargin, career win-rate, and even `mc_p`/`mc_o`. **The model extracts a
small, real signal from the text -- it just isn't a signal that improves calibrated predictions on
this TEST set once ELO/career/H2H are already present**, consistent with the paired-bootstrap
verdict above (small positive permutation importance and a small negative net effect on Brier are
not contradictory -- the model can learn to lean on a noisy signal in a way that adds variance
without adding calibrated accuracy).

**Honest bottom line:**

1. **Does the text prior help overall?** No. Point estimate is slightly worse (+0.00113 Brier,
   not significant but centered on the wrong side), confirmed by an independent same-pipeline
   NO_TEXT ablation (-0.00075 in the *other* direction, i.e. worse with text).
2. **Does it help cold-start players specifically?** No -- this is the more interesting negative
   result. The hypothesis predicted the text prior would help most where ELO/career stats are
   thinnest (very-low-history players). Instead, the very-low bucket shows the *largest*
   point-estimate degradation from adding text (+0.00253), not an improvement, and none of the
   three buckets show a statistically significant effect in either direction.
3. **Did the model use the feature at all?** Yes, modestly -- `text_skill` moved from provably
   inert (Exp C3, exact 0) to a small positive permutation importance (rank #17-18/27), so the
   text score is not simply ignored; it's a weak, partially-used signal that does not translate
   into better calibrated probabilities on this test set.

**Plausible explanations (not tested further in this notebook):** (a) the ELO/CareerWinRate/
MatchCount/Forma head block Exp C3 already carries most of the substitutable "how good is this
player" information, leaving little unique variance for a 1-10 self/LLM-assessed skill score to
add, even for newer players (their few matches' ELO movements may already be quite informative);
(b) self-reported free text (`Trenutci`) is a noisier, less calibrated signal than ELO/match
history even for players with almost no match history, since it reflects the player's own
subjective self-assessment rather than a competitive result; (c) 86% coverage with a neutral fill
for the rest introduces a genuine floor on how much signal this feature can carry, particularly in
the very-low bucket where n=1,076 is the smallest test subset and confidence intervals are
correspondingly widest. This project's original expectation -- "small effect given the ELO/career
substitutability pattern" -- held for the overall test, but the cold-start-specific hypothesis this
notebook was purpose-built to test did not pan out either, at least not on this text-scoring
methodology and this dataset size.


---

**Update (see `12_walkforward_with_c3_c4.ipynb`):** walk-forward CV across the 4 independent time folds nuances, but does not overturn, the single-split verdict above. Exp C4 actually beats Exp C3 in the two earliest folds (by a tiny margin, ≤0.0005 Brier) and loses to Exp C3 in the two most recent folds (matching this notebook's single-split finding almost exactly, since fold 4 IS this notebook's split) — the sign flips exactly at the fold-3/4 boundary. Every fold's C3-vs-C4 gap stays at or under 0.001 Brier, so `12`'s reading is that this is a genuinely noisy, sign-unstable, practically negligible effect rather than a confident "text never helps" result — softer than, but consistent with, the single-split framing above (no bootstrap CIs were re-run per fold in `12`, so this remains an open question rather than a settled one). What IS robust across all 4 folds: both Exp C3 and Exp C4 (the enriched-GRU family) occupy the top-2 spots in every single fold, decisively beating Exp B' even in the thinnest-data fold where Exp B' won outright in `09_walkforward_top_models.ipynb` — so whichever of C3/C4 a deployment ultimately prefers, the enrichment bundle itself (not the text feature specifically) is the durable, time-validated result, and this notebook's own "text is a rigorous null on this test set" reading should be understood as correct for the mature-data regime it was tested in, not as a universal, time-invariant claim.
